In [1]:
import sys
from pathlib import Path
from datetime import date

import pandas as pd

sys.path.append(
    str(Path("..").resolve())
)

from src.data_processing import (
    validate_schema,
    separate_valid_invalid_rows,
    prepare_data,
    add_dynamodb_keys
)


In [2]:
RAW_DATA_PATH = Path(
    "../data/raw/"
    "ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv"
)

CLEAN_DATA_PATH = Path(
    "../data/processed/hdb_resale_clean.csv"
)

DYNAMODB_DATA_PATH = Path(
    "../data/processed/hdb_resale_dynamodb.csv"
)

QUARANTINE_DATA_PATH = Path(
    "../data/processed/hdb_resale_quarantine.csv"
)

In [3]:
df_raw = pd.read_csv(
    RAW_DATA_PATH
)

print("Raw rows:", len(df_raw))
print("Columns:", df_raw.columns.tolist())

Raw rows: 244230
Columns: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price']


In [4]:
validate_schema(df_raw)

print("Schema validation passed.")

Schema validation passed.


In [5]:
df_valid, df_quarantine = (
    separate_valid_invalid_rows(
        df_raw
    )
)

print("Total rows:", len(df_raw))
print("Valid rows:", len(df_valid))
print(
    "Quarantined rows:",
    len(df_quarantine)
)

Total rows: 244230
Valid rows: 244230
Quarantined rows: 0


In [6]:
if len(df_quarantine) > 0:
    display(
        df_quarantine[
            [
                "month",
                "town",
                "flat_type",
                "validation_error"
            ]
        ].head(20)
    )

In [7]:
df_clean = prepare_data(
    df_valid
)

print(
    "Prepared rows:",
    len(df_clean)
)

Prepared rows: 244230


In [8]:
print(
    df_clean[
        [
            "storey_mid",
            "remaining_lease_months",
            "month_index"
        ]
    ].isna().sum()
)

storey_mid                0
remaining_lease_months    0
month_index               0
dtype: int64


In [9]:
print(
    "Date range:",
    df_clean["month"].min(),
    "to",
    df_clean["month"].max()
)

Date range: 2017-01-01 00:00:00 to 2026-11-01 00:00:00


In [10]:
SNAPSHOT_DATE = pd.Timestamp(
    date.today()
)

current_month = pd.Timestamp(
    year=SNAPSHOT_DATE.year,
    month=SNAPSHOT_DATE.month,
    day=1
)

latest_complete_month = (
    current_month
    - pd.offsets.MonthBegin(1)
)

snapshot_info = {
    "snapshot_date":
        SNAPSHOT_DATE.date().isoformat(),

    "row_count":
        len(df_clean),

    "min_month":
        df_clean["month"]
        .min()
        .strftime("%Y-%m"),

    "max_month_in_data":
        df_clean["month"]
        .max()
        .strftime("%Y-%m"),

    "latest_complete_month":
        latest_complete_month
        .strftime("%Y-%m")
}

snapshot_info

{'snapshot_date': '2026-11-05',
 'row_count': 244230,
 'min_month': '2017-01',
 'max_month_in_data': '2026-11',
 'latest_complete_month': '2026-10'}

In [11]:
df_complete = df_clean[
    df_clean["month"]
    <= latest_complete_month
].copy()

print(
    "All validated rows:",
    len(df_clean)
)

print(
    "Complete-month rows:",
    len(df_complete)
)

print(
    "Latest complete month:",
    df_complete["month"].max()
)

All validated rows: 244230
Complete-month rows: 243872
Latest complete month: 2026-10-01 00:00:00


In [12]:
duplicate_count = (
    df_clean.duplicated(
        subset=[
            "month",
            "town",
            "flat_type",
            "block",
            "street_name",
            "storey_range",
            "floor_area_sqm",
            "flat_model",
            "lease_commence_date",
            "remaining_lease",
            "resale_price"
        ]
    ).sum()
)

print(
    "Exact duplicate-looking rows:",
    duplicate_count
)

Exact duplicate-looking rows: 318


In [13]:
CLEAN_DATA_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

df_clean.to_csv(
    CLEAN_DATA_PATH,
    index=False
)

print(
    "Saved clean master:",
    CLEAN_DATA_PATH
)

Saved clean master: ..\data\processed\hdb_resale_clean.csv


In [14]:
if len(df_quarantine) > 0:

    df_quarantine.to_csv(
        QUARANTINE_DATA_PATH,
        index=False
    )

    print(
        "Saved quarantine file:",
        QUARANTINE_DATA_PATH
    )

In [15]:
df_dynamodb = (
    add_dynamodb_keys(
        df_clean
    )
)

In [16]:
print(
    "Duplicate flat_id:",
    df_dynamodb[
        "flat_id"
    ].duplicated().sum()
)

print(
    "Duplicate DynamoDB keys:",
    df_dynamodb.duplicated(
        subset=[
            "town_flat_type",
            "month_flat_id"
        ]
    ).sum()
)

Duplicate flat_id: 0
Duplicate DynamoDB keys: 0


In [17]:
df_dynamodb.to_csv(
    DYNAMODB_DATA_PATH,
    index=False
)

print(
    "Saved DynamoDB dataset:",
    DYNAMODB_DATA_PATH
)

Saved DynamoDB dataset: ..\data\processed\hdb_resale_dynamodb.csv
